# Phase 1 — Your First Anthropic API Call

Goal of this notebook: send a single message to Claude, get a response, and understand the shape of what comes back.

## What you'll learn
1. How to authenticate with the API using a `.env` file
2. The minimum required arguments for `client.messages.create()`
3. The full response object vs. the bit you usually care about (`message.content[0].text`)
4. What `stop_reason` and `usage` tell you

## Before you run anything
1. Copy `.env.example` to `.env` and paste your real API key
2. Install deps: `pip install -r ../requirements.txt`
3. Run the cells one at a time, top to bottom

## Step 1 — Load your API key from `.env`

Never hardcode the key in code. `python-dotenv` reads the `.env` file into environment variables, and the Anthropic SDK auto-picks up `ANTHROPIC_API_KEY` from there.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")  # one level up from notebooks/

import os
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not loaded — check your .env file"
print("Key loaded:", os.getenv("ANTHROPIC_API_KEY")[:12] + "...")

Key loaded: gela...


## Step 2 — Create the client and pick a model

The client is reusable across many requests. Define the model name once so you can swap it easily later.

In [3]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from environment
model = "claude-haiku-4-5"  # balanced intelligence + speed; swap for haiku/opus later

## Step 3 — Send your first request

Three required arguments:
- `model` — which Claude to use
- `max_tokens` — safety cap on output length (not a target)
- `messages` — the conversation so far

A `messages` list is a list of dicts with `role` (`user` or `assistant`) and `content`.

In [4]:
response = client.messages.create(
    model=model,
    max_tokens=5,
    messages=[
        {"role": "user", "content": "In two sentences, what is quantum computing?"}
    ],
)
response

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CbPiADvUiRvnJqcNuXpPz'}

## Step 4 — Pull out the parts you actually want

The full response object has metadata you usually ignore. The text lives at `response.content[0].text`.

**Why `content[0]` and not `content`?** Because the model can return *multiple blocks* per message (text, tool_use, thinking, etc.). A plain text reply is just a single text block. You'll see multi-block messages when you get to tool use.

In [1]:
print("Reply text:")
print(response.content[0].text)
print()
print("Stop reason:", response.stop_reason)   # 'end_turn' = model finished naturally
print("Usage:     ", response.usage)            # input + output token counts (and cost driver)

Reply text:


NameError: name 'response' is not defined

## Mini-exercise

1. Change the prompt to something where you'd expect Claude to hit the `max_tokens` cap. Set `max_tokens=20` and ask it to write a long essay. Check that `stop_reason` becomes `"max_tokens"` instead of `"end_turn"`.
2. Swap the model to `"claude-haiku-4-5"` (faster, cheaper) and ask the same question. Note any difference in speed and answer style.

When you've done both, ping me and we'll move to Phase 2 (multi-turn conversations + system prompts).